# 01 — Prepare (Fase 2): Profiling del dataset

**Objetivo:** entender el dataset *antes* de limpiarlo. Aquí solo exploramos y documentamos — la limpieza es la Fase 3 (notebook 02).

**Dataset:** Hotel Booking Demand · `../data/raw/hotel_bookings.csv` · 119.390 filas × 32 columnas · variable objetivo `is_canceled`.

**Contenido:** tamaño → tipos → nulos (conteo + %) → rangos numéricos → valores categóricos → duplicados → filas sin huéspedes → resumen de problemas.

Los hallazgos del profiling alimentan `docs/data_dictionary.md`.

## 1. Cargar el dataset

Cargamos el CSV crudo en un DataFrame de pandas; es la base de todo el profiling.

In [1]:
# Lee el CSV a un DataFrame con pandas (ruta relativa al notebook)
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")

## 2. Tamaño del dataset

Número de filas y columnas.

In [2]:

# Dimensiones del dataset: número de filas y columnas
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")


Number of rows: 119390
Number of columns: 32


## 3. Vista general de columnas

Qué columnas hay, de qué tipo y cuántos valores no nulos tiene cada una.

In [3]:
# Tipos de dato y número de valores no nulos por columna
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

## 4. Nulos

Cuántos nulos tiene cada columna y cuáles son las más afectadas.

In [4]:
df.isnull().sum()                  # nº de nulos (NaN) por columna

# nos encontramos que la columna country tiene 488 nulos, la columna children tiene 4 nulos, la columna agent tiene 16340 nulos y la columna company tiene 112593 nulos.

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

### % de nulos por columna (sobre el total de filas)

In [10]:
# % de nulos sobre el total de filas (solo las columnas con faltantes)
cols = ["agent","company","country","children"]
(df[cols].isna().mean()*100).round(4)

agent       13.6862
company     94.3069
country      0.4087
children     0.0034
dtype: float64

## 5. Rangos numéricos → outliers

Mira min, max y media de las columnas numéricas. ¿Hay algún valor **imposible**? (ej: `adr` negativo o gigante)

In [5]:
# Estadística descriptiva de las numéricas en una tabla. Sacamos valores para el diccionario de datos.
# .T transpone (cada columna como fila, más legible) · .round(2) = 2 decimales
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.37,0.48,0.00,0.00,0.00,1.0,1.0
lead_time,119390.0,104.01,106.86,0.00,18.00,69.00,160.0,737.0
arrival_date_year,119390.0,2016.16,0.71,2015.00,2016.00,2016.00,2017.0,2017.0
arrival_date_week_number,119390.0,27.17,13.61,1.00,16.00,28.00,38.0,53.0
arrival_date_day_of_month,119390.0,15.80,8.78,1.00,8.00,16.00,23.0,31.0
stays_in_weekend_nights,119390.0,0.93,1.00,0.00,0.00,1.00,2.0,19.0
stays_in_week_nights,119390.0,2.50,1.91,0.00,1.00,2.00,3.0,50.0
adults,119390.0,1.86,0.58,0.00,2.00,2.00,2.0,55.0
children,119386.0,0.10,0.40,0.00,0.00,0.00,0.0,10.0
babies,119390.0,0.01,0.10,0.00,0.00,0.00,0.0,10.0


### Valores únicos — columnas categóricas y binarias

Complementa el `describe()` numérico de arriba: cubre el resto de columnas (las no numéricas).

In [6]:
# Valores únicos de las columnas categóricas y binarias
for col in ['hotel', 'deposit_type', 'customer_type', 'reserved_room_type', 'assigned_room_type', 'reservation_status', 'is_canceled', 'is_repeated_guest']:
    print(f"=== {col} ===")
    print(df[col].unique())
    print()

=== hotel ===
<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str

=== deposit_type ===
<StringArray>
['No Deposit', 'Refundable', 'Non Refund']
Length: 3, dtype: str

=== customer_type ===
<StringArray>
['Transient', 'Contract', 'Transient-Party', 'Group']
Length: 4, dtype: str

=== reserved_room_type ===
<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'P', 'B']
Length: 10, dtype: str

=== assigned_room_type ===
<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'I', 'B', 'H', 'P', 'L', 'K']
Length: 12, dtype: str

=== reservation_status ===
<StringArray>
['Check-Out', 'Canceled', 'No-Show']
Length: 3, dtype: str

=== is_canceled ===
[0 1]

=== is_repeated_guest ===
[0 1]



## 6. Categóricas sospechosas

Valores únicos y frecuencias de algunas columnas de texto (`meal`, `distribution_channel`, `market_segment`, `country`). Buscamos categorías tipo `Undefined` que en realidad son datos faltantes disfrazados.

In [7]:
# Valores de las categóricas (ojo a "Undefined" = dato faltante disfrazado)
for col in ['meal', 'distribution_channel', 'market_segment', 'country']:
    print(f"=== {col} ===")
    print(df[col].value_counts(dropna=False))
    print()

=== meal ===
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64

=== distribution_channel ===
distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Undefined        5
Name: count, dtype: int64

=== market_segment ===
market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64

=== country ===
country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 178, dtype: int64



## 7. Duplicados

Cuántas filas son duplicados exactos. Este dataset no tiene ID único de reserva, así que un duplicado puede ser un error de exportación o dos reservas reales idénticas (se resuelve en la Fase 3).

In [8]:
# Nº de filas duplicadas exactas
print(df.duplicated().sum())              # cuántas filas duplicadas hay

# Ejemplos de duplicados (no se imprimen todas)
df[df.duplicated(keep=False)] 

# 31.994 duplicados exactos; sin ID único no se puede confirmar si son errores o reservas reales idénticas.

31994


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
21,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
39,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119352,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03
119353,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03
119354,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03
119372,City Hotel,0,175,2017,August,35,31,1,3,1,...,No Deposit,42.0,NaN,0,Transient,82.35,0,1,Check-Out,2017-09-04


## 8. Filas sin huéspedes → limitación

¿Cuántas filas tienen 0 personas? (`adults` + `children` + `babies` = 0)

In [9]:
# Filas sin ningún huésped (adults + children + babies = 0)
sin_huespedes = df[(df['adults'] + df['children'] + df['babies']) == 0]
print(len(sin_huespedes))

180


## 9. Resumen de problemas

Detectados en el profiling → se arreglan en **Fase 3** (cada uno = 1 fila del changelog):

| Problema | Detalle |
|---|---|
| Nulos reales (NaN) | company 112.593 (94 %), agent 16.340, country 488, children 4 |
| Nulos disfrazados ("Undefined") | meal 1.169, distribution_channel 5, market_segment 2 |
| Outliers | adr mín −6.38 (imposible) y máx 5.400; adults 55; children 10; babies 10; stays_in_week_nights 50 |
| Duplicados | 31.994 filas exactas (sin ID único → ambiguo) |
| Sin huéspedes | 180 filas con 0 adults+children+babies |
| Tipo incorrecto | reservation_status_date es texto → convertir a fecha |